In [10]:
import time
import json
import argparse
import logging
import os
from datetime import datetime, timezone
import requests
from dotenv import load_dotenv
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure

In [11]:
load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

In [12]:
ENDPOINTS = [
    {
        "name": "open-meteo",
        "category": "weather",
        "url": "https://api.open-meteo.com/v1/forecast",
        "params": {"latitude": 52.52, "longitude": 13.41, "current_weather": True},
        "provider": "open-meteo.com",
        "region_hint": "EU",
    },
    {
        "name": "exchangerate-usd",
        "category": "finance",
        "url": "https://open.er-api.com/v6/latest/USD",
        "params": {},
        "provider": "exchangerate-api.com",
        "region_hint": "US",
    },
    {
        "name": "ipapi",
        "category": "network",
        "url": "https://ipapi.co/json/",
        "params": {},
        "provider": "ipapi.co",
        "region_hint": "EU",
    },
    {
        "name": "rest-countries",
        "category": "reference",
        "url": "https://restcountries.com/v3.1/name/germany",
        "params": {"fields": "name,capital,population"},
        "provider": "restcountries.com",
        "region_hint": "EU",
    },
    {
        "name": "jsonplaceholder-posts",
        "category": "mock",
        "url": "https://jsonplaceholder.typicode.com/posts",
        "params": {},
        "provider": "jsonplaceholder.typicode.com",
        "region_hint": "US",
    },
    {
        "name": "catfact",
        "category": "trivial",
        "url": "https://catfact.ninja/fact",
        "params": {},
        "provider": "catfact.ninja",
        "region_hint": "US",
    },
    {
        "name": "agify",
        "category": "inference",
        "url": "https://api.agify.io/",
        "params": {"name": "michael"},
        "provider": "agify.io",
        "region_hint": "EU",
    },
    {
        "name": "dog-ceo",
        "category": "media",
        "url": "https://dog.ceo/api/breeds/list/all",
        "params": {},
        "provider": "dog.ceo",
        "region_hint": "US",
    },
]


In [13]:
TIMEOUT_SECONDS = 10
INTERVAL_SECONDS = 60 
BATCH_WRITE = True   

In [14]:
def ping(endpoint: dict) -> dict:
    """Ping one endpoint and return a structured metric record."""
    url = endpoint["url"]
    start = time.perf_counter()
    record = {
        "timestamp": datetime.now(timezone.utc),
        "endpoint": endpoint["name"],
        "category": endpoint["category"],
        "provider": endpoint["provider"],
        "region_hint": endpoint["region_hint"],
        "url": url,
    }
    try:
        resp = requests.get(url, params=endpoint.get("params", {}),
                            timeout=TIMEOUT_SECONDS)
        elapsed_ms = (time.perf_counter() - start) * 1000

        # DNS + TCP + TLS timing (only available if requests exposes it)
        try:
            dns_ms  = resp.elapsed.total_seconds() * 1000  # server-side only
        except Exception:
            dns_ms = None

        payload_bytes = len(resp.content)

        record.update({
            "status_code": resp.status_code,
            "success": resp.ok,
            "latency_ms": round(elapsed_ms, 2),
            "server_time_ms": round(dns_ms, 2) if dns_ms else None,
            "payload_bytes": payload_bytes,
            "content_type": resp.headers.get("content-type", ""),
            "error": None,
        })
    except requests.exceptions.Timeout:
        record.update({
            "status_code": None,
            "success": False,
            "latency_ms": TIMEOUT_SECONDS * 1000,
            "server_time_ms": None,
            "payload_bytes": 0,
            "content_type": "",
            "error": "timeout",
        })
    except requests.exceptions.RequestException as exc:
        elapsed_ms = (time.perf_counter() - start) * 1000
        record.update({
            "status_code": None,
            "success": False,
            "latency_ms": round(elapsed_ms, 2),
            "server_time_ms": None,
            "payload_bytes": 0,
            "content_type": "",
            "error": str(exc)[:200],
        })

    log.info("%-30s  %s  %7.1f ms  %d B",
             record["endpoint"],
             record["status_code"] or "ERR",
             record["latency_ms"],
             record["payload_bytes"])
    return record


In [15]:
def run(mongo_uri: str | None, limit: int | None):
    """Main collection loop."""
    collection = None
    if mongo_uri:
        try:
            client = MongoClient(mongo_uri, serverSelectionTimeoutMS=5000)
            client.admin.command("ping")
            db = client["api_monitor"]
            collection = db["latency_logs"]
            log.info("Connected to MongoDB Atlas ✓")
        except ConnectionFailure as exc:
            log.warning("MongoDB unavailable (%s) — writing to local JSONL fallback.", exc)

    fallback_path = "latency_logs.jsonl"
    total = 0

    while True:
        round_records = [ping(ep) for ep in ENDPOINTS]

        if collection is not None:
            try:
                collection.insert_many(round_records, ordered=False)
            except Exception as exc:
                log.warning("Mongo insert failed: %s", exc)
                collection = None   # fall back for rest of session

        if collection is None:
            with open(fallback_path, "a") as fh:
                for r in round_records:
                    r["timestamp"] = r["timestamp"].isoformat()
                    fh.write(json.dumps(r) + "\n")

        total += len(round_records)
        log.info("Round complete. Total records: %d", total)

        if limit and total >= limit:
            log.info("Limit %d reached — stopping.", limit)
            break

        time.sleep(INTERVAL_SECONDS)



In [ ]:
if __name__ == "__main__":
    import sys
    parser = argparse.ArgumentParser(description="API Latency Collector")
    parser.add_argument("--mongo-uri", default=os.getenv("MONGO_URI"))
    parser.add_argument("--limit", type=int, default=None)
    parser.add_argument("--interval", type=int, default=INTERVAL_SECONDS)

    args, _ = parser.parse_known_args()  # <-- ignore unknown args like Jupyter's -f

    INTERVAL_SECONDS = args.interval
    run(mongo_uri=args.mongo_uri, limit=args.limit)

22:37:26  INFO      open-meteo                      200    941.5 ms  472 B
22:37:26  INFO      exchangerate-usd                200    254.2 ms  2969 B
22:37:27  INFO      ipapi                           403    246.8 ms  5421 B
22:37:27  INFO      rest-countries                  200    799.5 ms  201 B
22:37:28  INFO      jsonplaceholder-posts           200   1042.6 ms  27520 B
22:37:29  INFO      catfact                         200    409.3 ms  157 B
22:37:31  INFO      agify                           200   1782.5 ms  42 B
22:37:31  INFO      dog-ceo                         200    347.7 ms  2445 B
22:37:31  INFO      Round complete. Total records: 8
22:38:32  INFO      open-meteo                      200    919.6 ms  472 B
22:38:32  INFO      exchangerate-usd                200     64.9 ms  2969 B
22:38:32  INFO      ipapi                           403     53.4 ms  5421 B
22:38:33  INFO      rest-countries                  200    739.6 ms  201 B
22:38:34  INFO      jsonplaceholder-posts